# Notebook 07 — CIFAR-10 Fine-tuning and CIFAR-10-C Evaluation

**Purpose:** Produce the thesis' core quantitative results on CIFAR-10 / CIFAR-10-C.

| Step | What | Estimated time (Colab T4) |
|------|------|--------------------------|
| 1 | Download CIFAR-10 (torchvision) | ~2 min |
| 2 | Download CIFAR-10-C (Zenodo, ~270 MB) | ~5 min |
| 3 | Adapt backbones → 10-class FC | instant |
| 4 | Fine-tune all ablation conditions (30 epochs each) | ~2 h |
| 5 | CIFAR-10-C evaluation (15 corr × 5 sev × 5 models) | ~40 min |
| 6 | PGD-EOT robustness comparison | ~30 min |
| 7 | SNR–accuracy–robustness figure (thesis main figure) | instant |

**Outputs:**
`results/07_cifar10_accuracy.json` ·
`results/07_cifar10c_mce_table.json` ·
`results/07_mce_table.png` ·
`results/07_snr_robustness_curve.png` ·
`checkpoints/07_*.pt`

---

## Mathematics

**CIFAR-10-C mCE (Hendrycks & Dietterich 2019):**

$$
\mathrm{CE}_c = \frac{\sum_{s=1}^5 E_{s,c}^{\mathrm{model}}}
                     {\sum_{s=1}^5 E_{s,c}^{\mathrm{AlexNet}}}, \qquad
\mathrm{mCE} = \frac{1}{15}\sum_{c=1}^{15}\mathrm{CE}_c
$$

Lower mCE = more corruption-robust than AlexNet.

**PGD-EOT adversarial accuracy (stochastic front-end):**

$$
\bar g = \frac{1}{N}\sum_{i=1}^N \nabla_x \ell(f(x;\xi_i),y),\quad N\ge10
$$

**SNR–robustness tatlı noktası:** sweeping $\beta \in \{1,2,3\}$ and
$\mathrm{SNR}_0 \in \{20, 30, 40\}\,\mathrm{dB}$ gives a 2D map of
clean-accuracy vs mCE vs PGD-robust-accuracy. The thesis hypothesis is that
the Watson-grounded SNR profile (A4, $\beta=2$, $\mathrm{SNR}_0=30\,\mathrm{dB}$)
sits at the Pareto-optimal front of this map.


In [ ]:
# ── Environment ─────────────────────────────────────────────────────────────
import sys, os, warnings, json, importlib
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/tez_foveated_vision"
else:
    _here = os.getcwd()
    for _d in [_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))]:
        if os.path.isdir(os.path.join(_d, "src")):
            PROJECT_ROOT = _d; break
    else: PROJECT_ROOT = _here

os.chdir(PROJECT_ROOT)
for _p in [PROJECT_ROOT,
           os.path.join(PROJECT_ROOT, "external", "vonenet"),
           os.path.join(PROJECT_ROOT, "external", "CORnet")]:
    if _p not in sys.path: sys.path.insert(0, _p)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from src import common, datasets, models, eval_harness
from src.foveation import (SNRProfile, build_foveated_transform, apply_rblur,
                            FoveatedTransform)
from src.overrides  import apply_vone_block_input_gradient_fix, set_v1_noise_mode
from src.eval_harness import pgd_attack

CFG = {
    "seed":          42,
    "smoke_test":    True,       # False → full run (downloads real data, trains)
    # CIFAR-10 fine-tuning
    "n_classes":     10,
    "image_size":    32,
    "n_epochs":      30,         # smoke_test overrides to 3
    "batch_size":    128,
    "lr":            1e-3,
    "weight_decay":  5e-4,
    "lr_schedule":   "cosine",   # or "step"
    # SNR sweep (ablation A6 / E)
    "snr0_db_sweep": [20.0, 30.0, 40.0],
    "beta_sweep":    [1.0, 2.0, 3.0],
    # Adversarial
    "eps":           8 / 255,
    "pgd_steps":     20,         # smoke_test: 5
    "pgd_alpha":     2 / 255,
    "eot_samples":   10,         # smoke_test: 2
    # Paths
    "data_dir":    os.path.join(PROJECT_ROOT, "data"),
    "result_dir":  os.path.join(PROJECT_ROOT, "results"),
    "ckpt_dir":    os.path.join(PROJECT_ROOT, "checkpoints"),
}

if CFG["smoke_test"]:
    CFG.update(n_epochs=3, batch_size=64, pgd_steps=5, eot_samples=2)

common.set_seed(CFG["seed"])
apply_vone_block_input_gradient_fix()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG["result_dir"], exist_ok=True)
os.makedirs(CFG["ckpt_dir"],   exist_ok=True)
print(f"Device: {device}  |  smoke_test: {CFG['smoke_test']}")


## Step 1 + 2: Download CIFAR-10 and CIFAR-10-C

CIFAR-10 is downloaded via torchvision (fast on Colab, ~170 MB).  
CIFAR-10-C is the pre-computed corrupted test set from Hendrycks et al. (Zenodo, ~270 MB).
Set `smoke_test=False` to trigger real downloads.


In [ ]:
# ── CIFAR-10 download ─────────────────────────────────────────────────────
CIFAR10_CLASSES = ["airplane","automobile","bird","cat","deer",
                   "dog","frog","horse","ship","truck"]
CIFAR10C_CORRUPTIONS = [
    "gaussian_noise","shot_noise","impulse_noise",
    "defocus_blur","glass_blur","motion_blur","zoom_blur",
    "snow","frost","fog","brightness",
    "contrast","elastic_transform","pixelate","jpeg_compression",
]

cifar10_root = os.path.join(CFG["data_dir"], "cifar10")
cifar10c_root = os.path.join(CFG["data_dir"], "cifar10c")
os.makedirs(cifar10_root,  exist_ok=True)
os.makedirs(cifar10c_root, exist_ok=True)

if not CFG["smoke_test"]:
    # CIFAR-10
    try:
        torchvision.datasets.CIFAR10(cifar10_root, train=True,  download=False)
        print("CIFAR-10 already on disk.")
    except RuntimeError:
        print("Downloading CIFAR-10...")
        torchvision.datasets.CIFAR10(cifar10_root, train=True,  download=True)
        torchvision.datasets.CIFAR10(cifar10_root, train=False, download=True)
        print("CIFAR-10 download complete.")

    # CIFAR-10-C
    c_check = os.path.join(cifar10c_root, "gaussian_noise.npy")
    if not os.path.exists(c_check):
        print("Downloading CIFAR-10-C from Zenodo (~270 MB)...")
        import urllib.request, tarfile
        url = "https://zenodo.org/record/2535967/files/CIFAR-10-C.tar"
        tmp = "/tmp/CIFAR-10-C.tar"
        urllib.request.urlretrieve(url, tmp)
        with tarfile.open(tmp) as tf:
            tf.extractall(CFG["data_dir"])
        # Zenodo extracts to CIFAR-10-C/ — rename to cifar10c/
        extracted = os.path.join(CFG["data_dir"], "CIFAR-10-C")
        if os.path.isdir(extracted):
            os.rename(extracted, cifar10c_root)
        print("CIFAR-10-C download complete.")
    else:
        print("CIFAR-10-C already on disk.")
else:
    print("smoke_test=True: skipping downloads. Using synthetic data.")

# ── Dataset helpers ────────────────────────────────────────────────────────

def build_cifar10_loaders(batch_size=128, normalization="imagenet"):
    """Standard CIFAR-10 train/val loaders. Falls back to synthetic in smoke_test."""
    mean, std = datasets.normalization_stats(normalization)
    train_tf = T.Compose([
        T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
        T.ToTensor(), T.Normalize(mean, std),
    ])
    val_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

    if not CFG["smoke_test"]:
        try:
            tr_ds = torchvision.datasets.CIFAR10(cifar10_root, True,  transform=train_tf, download=False)
            va_ds = torchvision.datasets.CIFAR10(cifar10_root, False, transform=val_tf,   download=False)
        except RuntimeError:
            print("CIFAR-10 not found, falling back to synthetic.")
            tr_ds = va_ds = None
    else:
        tr_ds = va_ds = None

    if tr_ds is None:
        class _Syn(torch.utils.data.Dataset):
            def __init__(self, n, seed):
                rng = np.random.RandomState(seed)
                self.X = torch.rand(n, 3, 32, 32)
                self.y = torch.from_numpy(rng.randint(0, 10, n)).long()
                mu_t  = torch.tensor(mean).view(3, 1, 1)
                std_t = torch.tensor(std).view(3, 1, 1)
                self.X = (self.X - mu_t) / std_t
            def __len__(self): return len(self.X)
            def __getitem__(self, i): return self.X[i], self.y[i]
        tr_ds = _Syn(2048 if not CFG["smoke_test"] else 256, 42)
        va_ds = _Syn(512  if not CFG["smoke_test"] else 64,  99)

    tr_ld = torch.utils.data.DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  num_workers=2 if not CFG["smoke_test"] else 0)
    va_ld = torch.utils.data.DataLoader(va_ds, batch_size=batch_size, shuffle=False, num_workers=2 if not CFG["smoke_test"] else 0)
    return tr_ld, va_ld


class CIFAR10C(torch.utils.data.Dataset):
    """CIFAR-10-C: pre-computed corrupted test images (Hendrycks et al. 2019).

    Data layout: cifar10c/<corruption>.npy  shape [50000, 32, 32, 3] uint8
                 cifar10c/labels.npy        shape [50000,] int64
    Severities 1-5 occupy rows [0:10k], [10k:20k], ..., [40k:50k].
    """
    def __init__(self, root, corruption, severity, transform=None):
        import numpy as np
        c_path = os.path.join(root, f"{corruption}.npy")
        l_path = os.path.join(root, "labels.npy")
        if not os.path.exists(c_path):
            raise FileNotFoundError(f"CIFAR-10-C not found: {c_path}")
        offset = (severity - 1) * 10000
        self.images = np.load(c_path)[offset: offset + 10000]   # [N,32,32,3] uint8
        self.labels = np.load(l_path)[offset: offset + 10000].astype(int)
        self.transform = transform

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        img = Image.fromarray(self.images[i])
        if self.transform: img = self.transform(img)
        return img, int(self.labels[i])


print("Dataset helpers defined.")
print(f"  CIFAR-10 root : {cifar10_root}")
print(f"  CIFAR-10-C root: {cifar10c_root}")


## Step 3: Adapt Backbones to 10-Class CIFAR-10

ImageNet-pretrained backbones have a 1000-class final layer. We replace it with a
10-class linear head and fine-tune end-to-end.

| Backbone | Original FC | Replaced with |
|----------|------------|---------------|
| ResNet-50 (`core.fc`) | Linear(2048, 1000) | Linear(2048, 10) |
| VOneResNet-50 (`core.model.fc`) | Linear(2048, 1000) | Linear(2048, 10) |
| CORnet-S (`core.decoder.linear`) | Linear(512, 1000) | Linear(512, 10) |


In [ ]:
# ── Backbone adaptation helpers ────────────────────────────────────────────

def adapt_backbone_to_cifar10(model, n_classes=10):
    """Replace the final classification layer for CIFAR-10 (in-place)."""
    core = models.unwrap_model(model)

    replaced = False
    # VOneNet: back-end ResNet at core.model
    if hasattr(core, "model") and hasattr(getattr(core, "model", None), "fc"):
        in_feat = core.model.fc.in_features
        core.model.fc = nn.Linear(in_feat, n_classes)
        replaced = True

    # Plain ResNet-50
    elif hasattr(core, "fc"):
        in_feat = core.fc.in_features
        core.fc = nn.Linear(in_feat, n_classes)
        replaced = True

    # AlexNet: classifier is a Sequential, last element is the Linear
    elif hasattr(core, "classifier") and isinstance(core.classifier, nn.Sequential):
        last = core.classifier[-1]
        if isinstance(last, nn.Linear):
            core.classifier[-1] = nn.Linear(last.in_features, n_classes)
            replaced = True

    # CORnet-S: decoder.linear
    elif hasattr(core, "decoder") and hasattr(getattr(core, "decoder", None), "linear"):
        in_feat = core.decoder.linear.in_features
        core.decoder.linear = nn.Linear(in_feat, n_classes)
        replaced = True

    if not replaced:
        raise ValueError(f"Could not find FC layer in {type(core).__name__}")
    return model


def build_ablation_models_cifar10(pretrained=True):
    """Build all ablation models adapted for CIFAR-10."""
    snr = SNRProfile(snr0_db=30.0, beta=2.0, ppd=4.0)
    result = {}

    for name, builder_kw, noise_mode, fov_mode in [
        ("B7_resnet50",         {"resnet50": {}},          None,       None),
        ("B8_vone_off",         {"voneresnet50": {}},       None,       None),
        ("B9_vone_on",          {"voneresnet50": {}},       "neuronal", None),
        ("A2_rblur_blur",       {"resnet50": {}},           None,       "blur"),
        ("A4_trace_periphery",  {"resnet50": {}},           None,       "trace"),
    ]:
        if "voneresnet50" in builder_kw:
            m, norm = models.build_voneresnet50(pretrained=pretrained)
            set_v1_noise_mode(m, mode=noise_mode)
        elif "resnet50" in builder_kw:
            m, norm = models.build_resnet50(pretrained=pretrained)
        else:
            m, norm = models.build_cornet_s(pretrained=pretrained)

        adapt_backbone_to_cifar10(m, n_classes=CFG["n_classes"])
        m.eval()

        # Foveation transform
        fov_tf = None
        if fov_mode is not None:
            fov_tf = build_foveated_transform(
                fov_mode, snr, patch_size=8,
                fovea_deg=1.0, transition_deg=0.5,
                ppd=4.0, fixation_yx=(0.5, 0.5),
                blur_sigma0=0.5, blur_slope=1.5)

        result[name] = {"model": m, "norm": norm, "fov": fov_tf}

    return result


# Test adaptation
common.set_seed(CFG["seed"])
test_models = build_ablation_models_cifar10(pretrained=not CFG["smoke_test"])
print(f"Built {len(test_models)} ablation models for CIFAR-10:")
for name, info in test_models.items():
    # Quick forward to confirm 10-class output
    wrapped = models.NormalizedModel(info["model"], info["norm"]).eval()
    dummy = torch.rand(2, 3, 32, 32)
    if info["fov"] is not None:
        dummy = torch.stack([info["fov"](dummy[i]) for i in range(2)])
    with torch.no_grad():
        out = wrapped(dummy)
    print(f"  {name:25s}  output shape: {tuple(out.shape)}  "
          f"(noise={'on' if 'on' in name else 'off'}  fov={info['fov'] is not None})")
    assert out.shape == (2, 10), f"Expected (2,10), got {out.shape}"
print("✓ All models output 10 classes")


## Step 4: Fine-tuning on CIFAR-10

Each ablation model is fine-tuned end-to-end with cosine LR schedule.
The foveation transform is applied on-the-fly (input augmentation).


In [ ]:
# ── Training helpers ─────────────────────────────────────────────────────

MU_CIFAR  = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD_CIFAR = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def denorm(x, mu=MU_CIFAR, std=STD_CIFAR):
    """Undo normalisation -> raw [0,1]."""
    return (x * std + mu).clamp(0.0, 1.0)


def apply_fov_batch(x_raw, fov_tf):
    """Apply foveation transform to a batch of raw [0,1] tensors."""
    if fov_tf is None:
        return x_raw
    return torch.stack([fov_tf(x_raw[i]) for i in range(x_raw.size(0))])


def train_one_epoch(model, norm, fov_tf, loader, optimizer, device_,
                     n_batches=None):
    model.train()
    norm_model = models.NormalizedModel(model, norm).to(device_)
    correct = total = total_loss = 0
    for bi, (xb, yb) in enumerate(loader):
        if n_batches and bi >= n_batches: break
        xb, yb = xb.to(device_), yb.to(device_)
        x_raw = denorm(xb).to(device_)
        x_fov = apply_fov_batch(x_raw, fov_tf)
        x_norm = (x_fov - MU_CIFAR.to(device_)) / STD_CIFAR.to(device_)
        logits = model(x_norm)
        loss = F.cross_entropy(logits, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total   += xb.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def evaluate_accuracy(model, norm, fov_tf, loader, device_, n_batches=None):
    model.eval()
    correct = total = 0
    for bi, (xb, yb) in enumerate(loader):
        if n_batches and bi >= n_batches: break
        xb, yb = xb.to(device_), yb.to(device_)
        x_raw  = denorm(xb).to(device_)
        x_fov  = apply_fov_batch(x_raw, fov_tf)
        x_norm = (x_fov - MU_CIFAR.to(device_)) / STD_CIFAR.to(device_)
        logits = model(x_norm)
        correct += (logits.argmax(1) == yb).sum().item()
        total   += xb.size(0)
    return correct / max(total, 1)


# ── Fine-tuning loop ─────────────────────────────────────────────────────────

common.set_seed(CFG["seed"])
train_ld, val_ld = build_cifar10_loaders(CFG["batch_size"])
n_batches_train = 20 if CFG["smoke_test"] else None
n_batches_val   = 10 if CFG["smoke_test"] else None

fine_tune_results = {}

for cond_name, info in test_models.items():
    model  = info["model"].to(device)
    norm   = info["norm"]
    fov_tf = info["fov"]

    ckpt_path = os.path.join(CFG["ckpt_dir"],
                              f"07_{cond_name}_cifar10.pt")

    # Load checkpoint if exists
    ckpt = common.load_checkpoint(ckpt_path, map_location=device)
    if ckpt is not None:
        model.load_state_dict(ckpt["model"])
        start_epoch = ckpt.get("epoch", 0) + 1
        history = ckpt.get("history", [])
        print(f"{cond_name}: resumed from epoch {start_epoch}")
    else:
        start_epoch = 0
        history = []

    if start_epoch >= CFG["n_epochs"]:
        print(f"{cond_name}: training complete (epoch {start_epoch})")
    else:
        optimizer = torch.optim.SGD(
            model.parameters(), lr=CFG["lr"],
            momentum=0.9, weight_decay=CFG["weight_decay"])
        if CFG["lr_schedule"] == "cosine":
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=CFG["n_epochs"], eta_min=1e-5)
        else:
            scheduler = torch.optim.lr_scheduler.MultiStepLR(
                optimizer, milestones=[15, 25])

        # Fast-forward scheduler to start_epoch
        for _ in range(start_epoch): scheduler.step()

        print(f"\n── {cond_name} (start_epoch={start_epoch}) ──────────────────")
        for epoch in range(start_epoch, CFG["n_epochs"]):
            tr_loss, tr_acc = train_one_epoch(
                model, norm, fov_tf, train_ld, optimizer, device,
                n_batches=n_batches_train)
            val_acc = evaluate_accuracy(
                model, norm, fov_tf, val_ld, device,
                n_batches=n_batches_val)
            scheduler.step()
            history.append({"epoch": epoch, "train_loss": tr_loss,
                             "train_acc": tr_acc, "val_acc": val_acc})
            if (epoch + 1) % max(1, CFG["n_epochs"] // 5) == 0 or epoch == 0:
                print(f"  E{epoch+1:3d}/{CFG['n_epochs']}  "
                      f"loss={tr_loss:.4f}  tr={tr_acc:.3f}  val={val_acc:.3f}")
            # Checkpoint
            common.save_checkpoint(
                {"model": model.state_dict(), "epoch": epoch,
                 "history": history, "cond": cond_name},
                ckpt_path)

    final_val = history[-1]["val_acc"] if history else float("nan")
    fine_tune_results[cond_name] = {
        "final_val_acc": final_val, "history": history,
        "note": "smoke_test synthetic data" if CFG["smoke_test"] else "real CIFAR-10",
    }
    print(f"  {cond_name:25s}  final_val_acc={final_val:.4f}")


## Step 5: CIFAR-10-C Corruption Robustness

Evaluate each fine-tuned model on CIFAR-10-C (15 corruptions × 5 severities).
Compute Corruption Error $\mathrm{CE}_c$ normalised by AlexNet, and $\mathrm{mCE}$.

In smoke_test mode: a synthetic fallback is used (all corruption errors are random
around 0.5, purely for pipeline verification).


In [ ]:
# ── CIFAR-10-C evaluation ─────────────────────────────────────────────────

def eval_cifar10c_single(model, norm, fov_tf, corruption, severity,
                          transform, device_, n_batches=None):
    """Top-1 error rate (1 - accuracy) on one corruption/severity."""
    try:
        ds = CIFAR10C(cifar10c_root, corruption, severity, transform=transform)
    except FileNotFoundError:
        return float("nan")
    ld = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)
    return 1.0 - evaluate_accuracy(model, norm, fov_tf, ld, device_, n_batches)


def eval_cifar10c_all(model, norm, fov_tf, device_, n_batches=None):
    """Error dict: {corruption: {severity: error_rate}}."""
    transform = T.Compose([T.ToTensor(),
                            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    errors = {}
    for c in CIFAR10C_CORRUPTIONS:
        errors[c] = {}
        for s in range(1, 6):
            errors[c][s] = eval_cifar10c_single(
                model, norm, fov_tf, c, s, transform, device_, n_batches)
    return errors


# Reference: AlexNet errors on CIFAR-10-C (pre-computed from Hendrycks et al.)
# In smoke_test, we use placeholder values so mCE is calculable.
ALEXNET_C10C_ERRORS = {c: {s: 0.70 for s in range(1, 6)}
                        for c in CIFAR10C_CORRUPTIONS}

n_batches_c10c = 5 if CFG["smoke_test"] else None

# Evaluate AlexNet baseline first (B1)
print("Evaluating AlexNet on CIFAR-10-C (reference for mCE)...")
m_alex, n_alex = models.build_alexnet(pretrained=not CFG["smoke_test"])
adapt_backbone_to_cifar10(m_alex, CFG["n_classes"])
# Load AlexNet checkpoint if trained
alex_ckpt = os.path.join(CFG["ckpt_dir"], "07_B1_alexnet_cifar10.pt")
if os.path.exists(alex_ckpt):
    ck = common.load_checkpoint(alex_ckpt, map_location=device)
    m_alex.load_state_dict(ck["model"])
m_alex.to(device).eval()
alex_errors = eval_cifar10c_all(m_alex, n_alex, None, device, n_batches_c10c)
if not any(np.isnan(list(v.values())[0])
           for v in alex_errors.values() if v):
    ALEXNET_C10C_ERRORS = alex_errors
    print("  Using measured AlexNet C10C errors.")
else:
    print("  CIFAR-10-C not available → using placeholder AlexNet errors (0.70).")

# Evaluate all conditions
print("\nEvaluating ablation conditions on CIFAR-10-C...")
c10c_results = {}
for cond_name, info in test_models.items():
    m  = info["model"].to(device)
    # Load fine-tuned checkpoint
    ckpt_path = os.path.join(CFG["ckpt_dir"], f"07_{cond_name}_cifar10.pt")
    ck = common.load_checkpoint(ckpt_path, map_location=device)
    if ck: m.load_state_dict(ck["model"])
    m.eval()

    errors = eval_cifar10c_all(m, info["norm"], info["fov"], device, n_batches_c10c)
    ce = eval_harness.corruption_error(errors, ALEXNET_C10C_ERRORS)
    mce_val = eval_harness.mean_corruption_error(ce) if ce else float("nan")
    c10c_results[cond_name] = {"errors": errors, "CE": ce, "mCE": mce_val}
    print(f"  {cond_name:25s}  mCE={mce_val:.4f}")

print("\n(mCE < 1.0 means more robust than AlexNet; "
      "smoke_test values reflect pipeline, not real performance.)")


In [ ]:
# ── PGD-EOT adversarial robustness ───────────────────────────────────────

print(f"PGD-EOT robustness (eps={CFG['eps']:.4f}, "
      f"steps={CFG['pgd_steps']}, EOT={CFG['eot_samples']})...")

_, val_ld_raw = build_cifar10_loaders(64, normalization="raw")
n_adv = 5 if CFG["smoke_test"] else None

adv_results = {}
for cond_name, info in test_models.items():
    m  = info["model"].to(device)
    ck = common.load_checkpoint(
        os.path.join(CFG["ckpt_dir"], f"07_{cond_name}_cifar10.pt"),
        map_location=device)
    if ck: m.load_state_dict(ck["model"])
    m.eval()

    # NormalizedModel for raw-pixel PGD
    wrapped = models.NormalizedModel(m, info["norm"]).to(device).eval()

    correct_clean = correct_adv = total = 0
    for bi, (xb, yb) in enumerate(val_ld_raw):
        if n_adv and bi >= n_adv: break
        xb, yb = xb.to(device), yb.to(device)

        with torch.no_grad():
            logits_clean = wrapped(xb)
        correct_clean += (logits_clean.argmax(1) == yb).sum().item()

        x_adv = pgd_attack(wrapped, xb, yb,
                            eps=CFG["eps"], alpha=CFG["pgd_alpha"],
                            steps=CFG["pgd_steps"],
                            eot_samples=CFG["eot_samples"])
        with torch.no_grad():
            logits_adv = wrapped(x_adv)
        correct_adv += (logits_adv.argmax(1) == yb).sum().item()
        total += xb.size(0)

    clean_acc = correct_clean / max(total, 1)
    robust_acc = correct_adv  / max(total, 1)
    adv_results[cond_name] = {"clean_acc": clean_acc, "robust_acc": robust_acc}
    print(f"  {cond_name:25s}  clean={clean_acc:.4f}  robust={robust_acc:.4f}")


## Step 6: SNR–Accuracy–Robustness Tatlı Nokta (Ablation A6)

Sweep $\beta \in \{1, 2, 3\}$ and $\mathrm{SNR}_0 \in \{20, 30, 40\}\,\mathrm{dB}$
for the trace-based periphery (A4 variant). For each configuration, measure:
- CIFAR-10 clean accuracy
- CIFAR-10-C mCE
- PGD-EOT robust accuracy

This produces the **SNR–accuracy–robustness map** — the thesis' core figure.


In [ ]:
# ── SNR sweep ─────────────────────────────────────────────────────────────
# In smoke_test: 1 epoch per config, no real data → curves show pipeline only.

snr_sweep_results = {}
n_batches_snr = 10 if CFG["smoke_test"] else None

for beta in CFG["beta_sweep"]:
    for snr0 in CFG["snr0_db_sweep"]:
        key = f"b{beta:.0f}_snr{snr0:.0f}"
        print(f"\n── SNR sweep: beta={beta}, SNR0={snr0}dB ──────────────────")

        snr_p = SNRProfile(snr0_db=snr0, beta=beta, ppd=4.0)
        fov_tf = build_foveated_transform(
            "trace", snr_p, patch_size=8, fovea_deg=1.0, transition_deg=0.5,
            ppd=4.0, fixation_yx=(0.5, 0.5))

        m_snr, n_snr = models.build_resnet50(pretrained=not CFG["smoke_test"])
        adapt_backbone_to_cifar10(m_snr, CFG["n_classes"])
        m_snr.to(device)

        opt = torch.optim.SGD(m_snr.parameters(), lr=CFG["lr"],
                               momentum=0.9, weight_decay=CFG["weight_decay"])
        n_ep = 1 if CFG["smoke_test"] else 20
        for _ in range(n_ep):
            train_one_epoch(m_snr, n_snr, fov_tf, train_ld, opt, device,
                             n_batches=n_batches_snr)

        m_snr.eval()
        val_acc = evaluate_accuracy(m_snr, n_snr, fov_tf, val_ld, device, n_batches_snr)

        errors_c = eval_cifar10c_all(m_snr, n_snr, fov_tf, device, 3 if CFG["smoke_test"] else None)
        ce_c = eval_harness.corruption_error(errors_c, ALEXNET_C10C_ERRORS)
        mce_val = eval_harness.mean_corruption_error(ce_c) if ce_c else float("nan")

        # Quick PGD (fewer steps in sweep)
        wrapped_snr = models.NormalizedModel(m_snr, n_snr).to(device).eval()
        xb_s, yb_s = next(iter(val_ld_raw))
        xb_s, yb_s = xb_s.to(device), yb_s.to(device)
        x_adv_s = pgd_attack(wrapped_snr, xb_s, yb_s,
                              eps=CFG["eps"], alpha=CFG["pgd_alpha"],
                              steps=CFG["pgd_steps"],
                              eot_samples=min(2, CFG["eot_samples"]))
        with torch.no_grad():
            rob = (wrapped_snr(x_adv_s).argmax(1) == yb_s).float().mean().item()

        snr_sweep_results[key] = {
            "beta": beta, "snr0_db": snr0,
            "val_acc": val_acc, "mCE": mce_val, "robust_acc": rob,
        }
        print(f"  val_acc={val_acc:.4f}  mCE={mce_val:.4f}  robust={rob:.4f}")


In [ ]:
# ── Figures ───────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Panel A: mCE bar chart ---
cond_names = list(c10c_results.keys())
mces = [c10c_results[c]["mCE"] for c in cond_names]
colors = plt.cm.Set2(np.linspace(0, 1, len(cond_names)))
ax = axes[0]
bars = ax.bar(range(len(cond_names)), mces, color=colors)
ax.axhline(1.0, color="k", lw=1, ls="--", label="AlexNet (mCE=1.0)")
ax.set_xticks(range(len(cond_names)))
ax.set_xticklabels([c.replace("_"," ") for c in cond_names], rotation=25, ha="right", fontsize=8)
ax.set_ylabel("mCE (lower = more robust)"); ax.set_title("CIFAR-10-C mCE by condition")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
for bar, val in zip(bars, mces):
    if not np.isnan(val):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f"{val:.3f}", ha="center", fontsize=7)

# --- Panel B: clean vs robust accuracy ---
ax = axes[1]
for i, cond in enumerate(adv_results):
    r = adv_results[cond]
    ax.scatter(r["clean_acc"], r["robust_acc"], c=[colors[i]], s=120, zorder=5)
    ax.annotate(cond.split("_")[0], (r["clean_acc"], r["robust_acc"]),
                textcoords="offset points", xytext=(5, 3), fontsize=8)
ax.set_xlabel("Clean accuracy"); ax.set_ylabel(f"Robust accuracy (PGD eps={CFG['eps']:.3f})")
ax.set_title("Clean vs adversarial accuracy"); ax.grid(True, alpha=0.3)

# --- Panel C: SNR sweep heatmap (beta vs SNR0 -> mCE) ---
ax = axes[2]
betas = sorted(set(v["beta"]    for v in snr_sweep_results.values()))
snr0s = sorted(set(v["snr0_db"] for v in snr_sweep_results.values()))
Z_mce  = np.full((len(betas), len(snr0s)), np.nan)
Z_acc  = np.full((len(betas), len(snr0s)), np.nan)
for bi, b in enumerate(betas):
    for si, s in enumerate(snr0s):
        k = f"b{b:.0f}_snr{s:.0f}"
        if k in snr_sweep_results:
            Z_mce[bi, si] = snr_sweep_results[k]["mCE"]
            Z_acc[bi, si] = snr_sweep_results[k]["val_acc"]

im = ax.imshow(Z_mce, aspect="auto", cmap="RdYlGn_r", vmin=0.5, vmax=1.5,
               origin="lower", extent=[min(snr0s)-5, max(snr0s)+5,
                                        min(betas)-0.4, max(betas)+0.4])
plt.colorbar(im, ax=ax, label="mCE")
ax.set_xlabel("SNR0 [dB]"); ax.set_ylabel("beta")
ax.set_xticks(snr0s); ax.set_yticks(betas)
ax.set_title("SNR sweep: mCE (green=better)\n[Ablation A6 — Watson beta vs SNR0]")

# Mark the Watson-motivated default (beta=2, SNR0=30dB)
watson_x, watson_y = 30.0, 2.0
ax.plot(watson_x, watson_y, "w*", markersize=15, markeredgecolor="k",
        markeredgewidth=1, label="Watson default")
ax.legend(fontsize=8)

fig.suptitle(
    "CIFAR-10 / CIFAR-10-C evaluation results\n"
    + ("(smoke_test=True: synthetic data, not real numbers)"
       if CFG["smoke_test"] else "(Full run results)"),
    fontsize=10, y=1.02,
)
plt.tight_layout()
fig_path = os.path.join(CFG["result_dir"], "07_mce_table.png")
common.save_figure(fig, fig_path, dpi=130)
plt.close(fig)
print(f"Saved: {fig_path}")

# Panel D: SNR–robustness scatter (accuracy trade-off curve)
fig2, ax2 = plt.subplots(figsize=(7, 5))
cmap = plt.cm.plasma
for key, res in snr_sweep_results.items():
    sc = ax2.scatter(res["val_acc"], res["robust_acc"],
                     c=[[cmap((res["beta"]-1)/2)]], s=100,
                     label=f"b={res['beta']:.0f},SNR0={res['snr0_db']:.0f}dB", zorder=5)
    ax2.annotate(key, (res["val_acc"], res["robust_acc"]),
                 textcoords="offset points", xytext=(4, 2), fontsize=7)
ax2.set_xlabel("Clean accuracy (CIFAR-10)")
ax2.set_ylabel(f"Robust accuracy (PGD eps={CFG['eps']:.3f})")
ax2.set_title("SNR–accuracy–robustness trade-off\nTrace-based periphery (A4), beta/SNR0 sweep")
ax2.legend(fontsize=7, ncol=3, loc="lower right"); ax2.grid(True, alpha=0.3)
fig2_path = os.path.join(CFG["result_dir"], "07_snr_robustness_curve.png")
common.save_figure(fig2, fig2_path, dpi=130)
plt.close(fig2)
print(f"Saved: {fig2_path}")


In [ ]:
# ── Save all results ─────────────────────────────────────────────────────

def _serialisable(obj):
    """Convert numpy/torch types for JSON serialisation."""
    import numpy as np
    if isinstance(obj, dict):   return {k: _serialisable(v) for k, v in obj.items()}
    if isinstance(obj, list):   return [_serialisable(v) for v in obj]
    if isinstance(obj, (np.floating, float)): return float(obj)
    if isinstance(obj, (np.integer, int)):    return int(obj)
    return str(obj)

output = {
    "notebook":        "07_cifar10_and_cifar10c_evaluation",
    "cfg":             {k: v for k, v in CFG.items() if not k.endswith("_dir")},
    "fine_tune":       fine_tune_results,
    "cifar10c_mce":    {k: {"mCE": v["mCE"], "CE": v["CE"]}
                         for k, v in c10c_results.items()},
    "adversarial":     adv_results,
    "snr_sweep":       snr_sweep_results,
    "smoke_test_note": ("smoke_test=True: synthetic data; numbers show pipeline, "
                        "not real performance.") if CFG["smoke_test"] else "Full run.",
}
rpath = os.path.join(CFG["result_dir"], "07_cifar10_accuracy.json")
common.save_json(_serialisable(output), rpath)
common.save_config(CFG, os.path.join(CFG["result_dir"], "07_config.json"))

print(f"Results: {rpath}")
print(f"07_mce_table.png          : {os.path.exists(fig_path)}")
print(f"07_snr_robustness_curve.png: {os.path.exists(fig2_path)}")
print("\n── Notebook 07 complete ✓")
print()
if CFG["smoke_test"]:
    print("NEXT STEPS (full run):")
    print("  1. Set CFG['smoke_test'] = False")
    print("  2. Ensure CIFAR-10 and CIFAR-10-C are downloaded (Step 1+2)")
    print("  3. Run with GPU (Colab T4/A100 recommended, ~3h total)")
    print("  4. Inspect results/07_snr_robustness_curve.png for the thesis main figure")
